# 04 - Rapid Spread Detection System

**Goal**: Build velocity + graph analysis to detect viral misinformation

**Approach**:
- Track claim velocity (appearances over time)
- Build social graph of claim propagation
- Detect coordinated inauthentic behavior
- Implement cooldown system for repeated claims

**Target**: Detect viral claims within 1 hour of spread

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import networkx as nx
from collections import defaultdict, Counter
import json

sns.set_style('whitegrid')
%matplotlib inline

## 1. Simulate Claim Spread Data

Since we don't have real-time spread data, we'll simulate it

In [ ]:
def generate_spread_data(n_claims=100, n_users=500, days=7):
    """Generate synthetic claim spread data"""
    data = []
    
    for claim_id in range(n_claims):
        # Some claims go viral, others don't
        is_viral = np.random.random() < 0.2
        
        if is_viral:
            # Viral: exponential growth
            n_shares = np.random.randint(50, 200)
            growth_rate = np.random.uniform(1.5, 3.0)
        else:
            # Normal: linear growth
            n_shares = np.random.randint(5, 30)
            growth_rate = 1.0
        
        start_time = datetime.now() - timedelta(days=np.random.randint(0, days))
        
        for i in range(n_shares):
            # Time between shares
            if is_viral:
                hours_offset = np.random.exponential(2)  # Fast spread
            else:
                hours_offset = np.random.exponential(12)  # Slow spread
            
            timestamp = start_time + timedelta(hours=hours_offset * i)
            user_id = np.random.randint(0, n_users)
            
            data.append({
                'claim_id': claim_id,
                'user_id': user_id,
                'timestamp': timestamp,
                'is_viral': is_viral
            })
    
    return pd.DataFrame(data)

df = generate_spread_data()
df = df.sort_values('timestamp').reset_index(drop=True)
print(f"Generated {len(df)} spread events for {df['claim_id'].nunique()} claims")
df.head()

## 2. Velocity Detection

Calculate claim velocity (shares per hour)

In [ ]:
def calculate_velocity(df, window_hours=1):
    """Calculate claim velocity in sliding window"""
    velocities = []
    
    for claim_id in df['claim_id'].unique():
        claim_df = df[df['claim_id'] == claim_id].sort_values('timestamp')
        
        if len(claim_df) < 2:
            continue
        
        # Calculate velocity in 1-hour windows
        start_time = claim_df['timestamp'].min()
        end_time = claim_df['timestamp'].max()
        
        current_time = start_time
        while current_time < end_time:
            window_end = current_time + timedelta(hours=window_hours)
            
            shares_in_window = len(claim_df[
                (claim_df['timestamp'] >= current_time) & 
                (claim_df['timestamp'] < window_end)
            ])
            
            velocities.append({
                'claim_id': claim_id,
                'timestamp': current_time,
                'velocity': shares_in_window / window_hours,
                'is_viral': claim_df['is_viral'].iloc[0]
            })
            
            current_time += timedelta(hours=window_hours)
    
    return pd.DataFrame(velocities)

velocity_df = calculate_velocity(df)
print(f"Calculated velocity for {len(velocity_df)} time windows")

# Plot velocity distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
viral = velocity_df[velocity_df['is_viral']]['velocity']
normal = velocity_df[~velocity_df['is_viral']]['velocity']
plt.hist([normal, viral], bins=30, label=['Normal', 'Viral'], alpha=0.7)
plt.xlabel('Velocity (shares/hour)')
plt.ylabel('Frequency')
plt.legend()
plt.title('Velocity Distribution')

plt.subplot(1, 2, 2)
plt.boxplot([normal, viral], labels=['Normal', 'Viral'])
plt.ylabel('Velocity (shares/hour)')
plt.title('Velocity Comparison')

plt.tight_layout()
plt.show()

print(f"\nViral velocity: {viral.mean():.2f} ± {viral.std():.2f} shares/hour")
print(f"Normal velocity: {normal.mean():.2f} ± {normal.std():.2f} shares/hour")

## 3. Graph-Based Spread Detection

Build social graph to detect coordinated behavior

In [ ]:
def build_spread_graph(df, claim_id):
    """Build propagation graph for a claim"""
    claim_df = df[df['claim_id'] == claim_id].sort_values('timestamp')
    
    G = nx.DiGraph()
    
    # Add nodes
    for user_id in claim_df['user_id'].unique():
        G.add_node(user_id)
    
    # Add edges (user A -> user B if B shared after A)
    users = claim_df['user_id'].tolist()
    for i in range(len(users) - 1):
        G.add_edge(users[i], users[i+1])
    
    return G

def detect_coordinated_behavior(G, threshold=3):
    """Detect if graph shows coordinated inauthentic behavior"""
    # Check for suspicious patterns
    
    # 1. High clustering (users sharing in tight groups)
    if len(G.nodes()) > 2:
        clustering = nx.average_clustering(G.to_undirected())
    else:
        clustering = 0
    
    # 2. Hub nodes (users sharing to many others)
    out_degrees = [d for n, d in G.out_degree()]
    max_out_degree = max(out_degrees) if out_degrees else 0
    
    # 3. Connected components (isolated groups)
    n_components = nx.number_weakly_connected_components(G)
    
    is_coordinated = (
        clustering > 0.5 or 
        max_out_degree > threshold or
        n_components > len(G.nodes()) * 0.3
    )
    
    return {
        'is_coordinated': is_coordinated,
        'clustering': clustering,
        'max_out_degree': max_out_degree,
        'n_components': n_components
    }

# Analyze a few claims
sample_claims = df['claim_id'].unique()[:5]

for claim_id in sample_claims:
    G = build_spread_graph(df, claim_id)
    result = detect_coordinated_behavior(G)
    
    print(f"\nClaim {claim_id}:")
    print(f"  Nodes: {len(G.nodes())}, Edges: {len(G.edges())}")
    print(f"  Coordinated: {result['is_coordinated']}")
    print(f"  Clustering: {result['clustering']:.3f}")
    print(f"  Max out-degree: {result['max_out_degree']}")

## 4. Cooldown System

Implement cooldown to avoid re-analyzing same claims

In [ ]:
class ClaimCooldownSystem:
    """Track analyzed claims and implement cooldown"""
    
    def __init__(self, cooldown_hours=24):
        self.cooldown_hours = cooldown_hours
        self.claim_cache = {}  # claim_hash -> {timestamp, result}
    
    def get_claim_hash(self, text):
        """Generate hash for claim text"""
        import hashlib
        return hashlib.sha256(text.encode()).hexdigest()[:16]
    
    def should_analyze(self, text):
        """Check if claim should be analyzed or is in cooldown"""
        claim_hash = self.get_claim_hash(text)
        
        if claim_hash not in self.claim_cache:
            return True, None
        
        cached = self.claim_cache[claim_hash]
        time_since = datetime.now() - cached['timestamp']
        
        if time_since.total_seconds() / 3600 > self.cooldown_hours:
            # Cooldown expired
            return True, cached['result']
        else:
            # Still in cooldown
            return False, cached['result']
    
    def cache_result(self, text, result):
        """Cache analysis result"""
        claim_hash = self.get_claim_hash(text)
        self.claim_cache[claim_hash] = {
            'timestamp': datetime.now(),
            'result': result
        }
    
    def get_stats(self):
        """Get cache statistics"""
        active = sum(
            1 for v in self.claim_cache.values()
            if (datetime.now() - v['timestamp']).total_seconds() / 3600 < self.cooldown_hours
        )
        return {
            'total_cached': len(self.claim_cache),
            'active_cooldowns': active
        }

# Test cooldown system
cooldown = ClaimCooldownSystem(cooldown_hours=24)

test_claims = [
    "COVID vaccines contain microchips",
    "Earth is flat",
    "COVID vaccines contain microchips",  # Duplicate
]

for claim in test_claims:
    should_analyze, cached_result = cooldown.should_analyze(claim)
    
    if should_analyze:
        print(f"Analyzing: {claim}")
        result = {'verdict': 'fake', 'confidence': 0.95}
        cooldown.cache_result(claim, result)
    else:
        print(f"Cooldown active: {claim} -> {cached_result}")

print(f"\nCache stats: {cooldown.get_stats()}")

## 5. Real-Time Alert System

Combine velocity + graph analysis for alerts

In [ ]:
class ViralDetectionSystem:
    """Real-time viral misinformation detection"""
    
    def __init__(self, velocity_threshold=10, cooldown_hours=24):
        self.velocity_threshold = velocity_threshold
        self.cooldown = ClaimCooldownSystem(cooldown_hours)
        self.alerts = []
    
    def analyze_claim(self, claim_text, spread_data):
        """Analyze if claim is going viral"""
        
        # Check cooldown
        should_analyze, cached = self.cooldown.should_analyze(claim_text)
        if not should_analyze:
            return cached
        
        # Calculate velocity
        velocity = len(spread_data) / 1.0  # shares per hour
        
        # Build graph
        G = nx.DiGraph()
        for i, user in enumerate(spread_data):
            G.add_node(user)
            if i > 0:
                G.add_edge(spread_data[i-1], user)
        
        coord_result = detect_coordinated_behavior(G)
        
        # Determine alert level
        is_viral = velocity > self.velocity_threshold
        is_coordinated = coord_result['is_coordinated']
        
        if is_viral and is_coordinated:
            alert_level = 'CRITICAL'
        elif is_viral:
            alert_level = 'HIGH'
        elif is_coordinated:
            alert_level = 'MEDIUM'
        else:
            alert_level = 'LOW'
        
        result = {
            'alert_level': alert_level,
            'velocity': velocity,
            'is_coordinated': is_coordinated,
            'graph_metrics': coord_result
        }
        
        # Cache result
        self.cooldown.cache_result(claim_text, result)
        
        # Log alert
        if alert_level in ['CRITICAL', 'HIGH']:
            self.alerts.append({
                'timestamp': datetime.now(),
                'claim': claim_text[:50],
                'level': alert_level,
                'velocity': velocity
            })
        
        return result

# Test system
detector = ViralDetectionSystem(velocity_threshold=10)

test_scenarios = [
    ('Normal claim', list(range(5))),  # 5 shares
    ('Viral claim', list(range(50))),  # 50 shares
    ('Coordinated claim', [1, 1, 1, 2, 2, 2, 3, 3, 3]),  # Repeated users
]

for claim, spread_data in test_scenarios:
    result = detector.analyze_claim(claim, spread_data)
    print(f"\n{claim}:")
    print(f"  Alert: {result['alert_level']}")
    print(f"  Velocity: {result['velocity']:.1f} shares/hour")
    print(f"  Coordinated: {result['is_coordinated']}")

print(f"\nTotal alerts: {len(detector.alerts)}")

## 6. Integration with Backend

Code to integrate into production system

In [ ]:
# Save detector for production use
import pickle

# In production, this would be a Redis/database-backed system
production_config = {
    'velocity_threshold': 10,  # shares per hour
    'cooldown_hours': 24,
    'graph_clustering_threshold': 0.5,
    'alert_levels': {
        'CRITICAL': 'velocity > 20 AND coordinated',
        'HIGH': 'velocity > 10',
        'MEDIUM': 'coordinated',
        'LOW': 'normal'
    }
}

with open('../backend/data/spread_detection_config.json', 'w') as f:
    json.dump(production_config, f, indent=2)

print("Spread detection config saved!")
print("\nNext steps:")
print("1. Implement in backend/app/analysis/platform_tracker.py")
print("2. Add Redis for claim cache")
print("3. Setup real-time monitoring dashboard")
print("4. Configure alert webhooks")

## Summary

Built rapid spread detection system with:
- Velocity tracking (shares per hour)
- Graph-based coordinated behavior detection
- Cooldown system to avoid re-analysis
- Multi-level alert system (CRITICAL/HIGH/MEDIUM/LOW)

**Production TODO**:
- [ ] Integrate with Redis for distributed caching
- [ ] Add real-time stream processing (Kafka/RabbitMQ)
- [ ] Build monitoring dashboard
- [ ] Setup alert webhooks (Slack/email)
- [ ] Collect real spread data from social platforms